**Séance :** 02  
**Nom :** FANOU Précieux  
**Porteur # :** P2  
**Étape assignée :** 2.2 --> 2.5

# Étape 2.2 : La barre de bruit

**Question** : quand la MAE change d'un essai à l'autre, est-ce une **vraie amélioration** ou seulement du **hasard** ?

À l'étape 1.10, on a mesuré un plancher sur **un seul** découpage 80/20. Avec un autre tirage au hasard, on aurait obtenu un autre chiffre. La **barre de bruit** est la taille de cette variation : un gain plus petit qu'elle n'est pas démontré.

| Ce qu'on mesure | Comment |
|---|---|
| La dispersion de la MAE selon le découpage | 20 découpages 80/20 stratifiés, avec 20 graines différentes |
| Une deuxième estimation du bruit | l'écart-type des erreurs, divisé par la racine du nombre de lignes |
| Une règle de décision | un gain n'est retenu que s'il dépasse 2 fois la barre |

> **Ce qui est une convention de ce notebook** : le facteur 2 (environ 95 % de confiance si le bruit est à peu près normal). Ce n'est pas une règle imposée par l'énoncé. L'énoncé précise seulement que la métrique de coût est l'erreur absolue.

## Bloc 1 : imports et chemins des fichiers

In [1]:
# ── Ce que fait ce bloc : importer les outils et trouver où est train.csv ──
from pathlib import Path                              # chemins de fichiers
import numpy as np                                    # calcul sur les tableaux de nombres
import pandas as pd                                   # lecture du CSV et tableaux
from sklearn.model_selection import train_test_split  # découpage de validation

# Sur Kaggle, les données sont dans /kaggle/input/... ; en local, dans le dossier courant
RACINE_KAGGLE = Path("/kaggle/input")

if RACINE_KAGGLE.exists():
    # On cherche train.csv UNIQUEMENT dans le jeu du sujet 04
    trouves = [p for p in RACINE_KAGGLE.rglob("train.csv") if "sujet-04" in str(p)]
    if not trouves:
        raise FileNotFoundError("train.csv du sujet 04 introuvable : ajoutez votre jeu (Add Input).")
    DOSSIER = trouves[0].parent
else:
    DOSSIER = Path(".")

CHEMIN_TRAIN = DOSSIER / "train.csv"

GRAINE_REFERENCE = 42      # la graine utilisée à l'étape 1.10 (plancher à 1810.40)
NB_REPETITIONS = 20        # nombre de découpages différents
FACTEUR = 2                # un gain doit dépasser FACTEUR x la barre de bruit

print("Dossier des données :", DOSSIER)
print("train.csv existe    :", CHEMIN_TRAIN.exists())

# ── Résultat attendu ──
# Dossier des données : /kaggle/input/datasets/prcieuxfanou/esgc-vak-sujet-04-fanou   (en local : .)
# train.csv existe    : True

Dossier des données : .
train.csv existe    : True


## Bloc 2 : lire `id` et `loss`, et préparer les tranches

In [2]:
# ── Ce que fait ce bloc : charger les deux colonnes utiles et ranger loss en 10 tranches ──
train = pd.read_csv(CHEMIN_TRAIN, usecols=["id", "loss"])   # seules ces deux colonnes servent ici
train["loss"] = train["loss"].astype("float64")
loss = train["loss"]

def mae(vraies, prediction):
    """Erreur absolue moyenne : de combien d'euros se trompe-t-on en moyenne ?"""
    return float(np.mean(np.abs(vraies - prediction)))

# 10 tranches de montant (déciles) : elles servent à stratifier, comme à l'étape 1.10
tranches = pd.qcut(loss, q=10, labels=False, duplicates="drop")

print("Forme :", train.shape)
print("Nombre de tranches :", tranches.nunique())

# ── Résultat attendu ──
# Forme : (150654, 2)
# Nombre de tranches : 10

Forme : (150654, 2)
Nombre de tranches : 10


## Bloc 3 : refaire le découpage 20 fois

Pour chaque graine, on coupe `train` en 80 % / 20 % (stratifié sur `loss`), on calcule la constante sur les 80 % **seulement**, et on mesure la MAE sur les 20 % cachés. Seule la graine change.

In [3]:
# ── Ce que fait ce bloc : une répétition du plancher, puis 20 répétitions ──
def repetition(graine):
    """Un découpage 80/20 stratifié, puis la MAE de la moyenne et de la médiane sur les 20 %."""
    indices_app, indices_val = train_test_split(
        train.index, test_size=0.20, random_state=graine, stratify=tranches
    )
    loss_app = loss.loc[indices_app]     # sert à calculer la constante
    loss_val = loss.loc[indices_val]     # caché : sert seulement à mesurer

    mediane_app = float(loss_app.median())
    moyenne_app = float(loss_app.mean())
    return {
        "graine": graine,
        "lignes_validation": len(loss_val),
        "ids_communs": len(set(indices_app) & set(indices_val)),   # doit rester à 0 (pas de fuite)
        "mae_mediane": mae(loss_val, mediane_app),
        "mae_moyenne": mae(loss_val, moyenne_app),
    }

# 20 graines différentes : 0, 1, ..., 19
resultats = pd.DataFrame([repetition(g) for g in range(NB_REPETITIONS)])
resultats["gain_mediane"] = resultats["mae_moyenne"] - resultats["mae_mediane"]

# La répétition de référence (graine 42, celle de l'étape 1.10)
reference = repetition(GRAINE_REFERENCE)

display(resultats.round(2))
print("MAE de la médiane, graine de référence 42 :", round(reference["mae_mediane"], 2))

# ── Résultat attendu ──
# Un tableau de 20 lignes. La MAE de la médiane varie un peu d'une ligne à l'autre
# (c'est exactement le bruit qu'on veut mesurer) et reste toujours inférieure à celle de la moyenne.
# La graine 42 redonne le plancher de l'étape 1.10 (1810.40 sur vos données).

,graine,lignes_validation,ids_communs,mae_mediane,mae_moyenne,gain_mediane
0,0,30131,0,1805.61,1963.18,157.57
1,1,30131,0,1807.02,1965.13,158.11
2,2,30131,0,1809.49,1966.88,157.38
3,3,30131,0,1810.65,1968.43,157.78
4,4,30131,0,1817.61,1976.00,158.40
5,5,30131,0,1810.07,1967.66,157.59
6,6,30131,0,1810.84,1967.63,156.78
7,7,30131,0,1796.65,1955.85,159.20
8,8,30131,0,1812.38,1969.59,157.21
9,9,30131,0,1820.22,1976.51,156.29


MAE de la médiane, graine de référence 42 : 1810.4


## Bloc 4 : mesurer la barre de bruit

In [4]:
# ── Ce que fait ce bloc : résumer la dispersion des 20 MAE ──
serie = resultats["mae_mediane"]

moyenne_mae = float(serie.mean())             # centre des 20 mesures
barre_bruit = float(serie.std(ddof=1))        # écart-type : LA barre de bruit
minimum, maximum = float(serie.min()), float(serie.max())

print(f"MAE de la médiane, moyenne des {NB_REPETITIONS} découpages : {moyenne_mae:.2f}")
print(f"Barre de bruit (écart-type)        : {barre_bruit:.2f} euros")
print(f"Minimum / maximum observés          : {minimum:.2f} / {maximum:.2f}")
print(f"Étendue (maximum - minimum)         : {maximum - minimum:.2f} euros")
print(f"Barre de bruit en % de la MAE       : {barre_bruit / moyenne_mae * 100:.2f} %")

# Le gain de la médiane sur la moyenne varie-t-il aussi ?
print()
print(f"Gain de la médiane sur la moyenne : moyenne {resultats['gain_mediane'].mean():.2f}, "
      f"écart-type {resultats['gain_mediane'].std(ddof=1):.2f}")
print("La médiane gagne sur la moyenne dans",
      int((resultats["gain_mediane"] > 0).sum()), "découpages sur", NB_REPETITIONS)

# ── Résultat attendu ──
# Une barre de bruit strictement positive, petite devant la MAE (à lire : on ne la suppose pas).
# La médiane gagne sur la moyenne dans les 20 découpages sur 20.

MAE de la médiane, moyenne des 20 découpages : 1812.13
Barre de bruit (écart-type)        : 6.40 euros
Minimum / maximum observés          : 1796.65 / 1823.77
Étendue (maximum - minimum)         : 27.12 euros
Barre de bruit en % de la MAE       : 0.35 %

Gain de la médiane sur la moyenne : moyenne 157.45, écart-type 0.91
La médiane gagne sur la moyenne dans 20 découpages sur 20


## Bloc 5 : une deuxième estimation du bruit

Sur **un seul** découpage, on peut aussi estimer le bruit de la MAE avec une formule : l'écart-type des erreurs absolues, divisé par la racine du nombre de lignes. Si les deux estimations sont du même ordre, on a plus confiance dans la barre.

In [5]:
# ── Ce que fait ce bloc : estimer le bruit par la formule, sur le découpage de référence ──
indices_app, indices_val = train_test_split(
    train.index, test_size=0.20, random_state=GRAINE_REFERENCE, stratify=tranches
)
mediane_ref = float(loss.loc[indices_app].median())     # constante apprise sur les 80 %
erreurs_val = np.abs(loss.loc[indices_val] - mediane_ref)   # erreur absolue de chaque sinistre caché

n_val = len(erreurs_val)
erreur_type_formule = float(erreurs_val.std(ddof=1) / np.sqrt(n_val))   # écart-type / racine(n)

print("Lignes de validation                       :", n_val)
print(f"MAE sur ce découpage                       : {erreurs_val.mean():.2f}")
print(f"Bruit estimé par la formule                : {erreur_type_formule:.2f} euros")
print(f"Barre de bruit mesurée sur 20 découpages   : {barre_bruit:.2f} euros")

# ── Résultat attendu ──
# Deux nombres du même ordre de grandeur, mais pas identiques : la formule suppose un tirage indépendant,
# alors que les 20 découpages piochent dans les mêmes 150 654 lignes et sont stratifiés sur loss.
# Il est possible que la mesure sur 20 découpages soit plus petite que la formule. Lisez les deux, ne les supposez pas.

Lignes de validation                       : 30131
MAE sur ce découpage                       : 1810.40
Bruit estimé par la formule                : 13.82 euros
Barre de bruit mesurée sur 20 découpages   : 6.40 euros


## Bloc 6 : la règle de décision

Un gain n'est retenu que s'il dépasse **FACTEUR x barre de bruit**. On écrit une petite fonction réutilisable pour tous les essais du bloc 2 du projet. Les trois MAE de modèle ci-dessous sont **inventées**, juste pour montrer comment la fonction répond.

In [6]:
# ── Ce que fait ce bloc : une fonction qui dit si un gain dépasse le bruit ──
plancher_ref = reference["mae_mediane"]   # plancher honnête (étape 1.10), découpage de référence
seuil_gain = FACTEUR * barre_bruit        # gain minimal pour être pris au sérieux

def gain_demontre(mae_modele, plancher=plancher_ref, seuil=seuil_gain):
    """Renvoie le gain en euros, et True s'il dépasse le seuil de bruit."""
    gain = plancher - mae_modele
    return gain, bool(gain > seuil)

print(f"Plancher de référence : {plancher_ref:.2f}")
print(f"Seuil de gain ({FACTEUR} x barre) : {seuil_gain:.2f} euros")
print()

# Exemples FICTIFS : trois MAE de modèle inventées, pour illustrer
exemples = []
for nom, valeur in [("essai fictif A (gain très faible)", plancher_ref - 0.5 * seuil_gain),
                    ("essai fictif B (gain juste au-dessus)", plancher_ref - 1.2 * seuil_gain),
                    ("essai fictif C (gain net)", plancher_ref - 5 * seuil_gain)]:
    gain, retenu = gain_demontre(valeur)
    exemples.append({"essai": nom, "MAE": round(valeur, 2), "gain": round(gain, 2),
                     "gain démontré ?": "oui" if retenu else "non"})
display(pd.DataFrame(exemples))

# ── Résultat attendu ──
# L'essai A est "non" (son gain est dans le bruit), les essais B et C sont "oui".

Plancher de référence : 1810.40
Seuil de gain (2 x barre) : 12.81 euros



,essai,MAE,gain,gain démontré ?
0,essai fictif A (gain très faible),1804.00,6.40,non
1,essai fictif B (gain juste au-dessus),1795.03,15.37,oui
2,essai fictif C (gain net),1746.37,64.03,oui


## Bloc 7 : tableau récapitulatif de l'étape 2.2

In [7]:
# ── Ce que fait ce bloc : contrôles internes, puis résultat à retenir ──
def joli(v):
    """Affiche 150654 plutôt que 150654.00."""
    return str(int(v)) if float(v).is_integer() else str(round(float(v), 2))

ecart_centre = abs(moyenne_mae / mae(loss, float(loss.median())) - 1)   # MAE moyenne des 20 vs MAE sur tout train

controles = [
    ("Lignes de train",                                     150654,           len(train)),
    ("Nombre de découpages",                                NB_REPETITIONS,   len(resultats)),
    ("Tailles de validation distinctes",                    1,                int(resultats["lignes_validation"].nunique())),
    ("id communs apprentissage / validation (total)",       0,                int(resultats["ids_communs"].sum())),
    ("Découpages où la médiane bat la moyenne",             NB_REPETITIONS,   int((resultats["gain_mediane"] > 0).sum())),
    ("Barre de bruit strictement positive (1 = oui)",       1,                int(barre_bruit > 0)),
    ("MAE moyenne des 20 à moins de 1 % de la MAE sur tout train (1 = oui)", 1, int(ecart_centre < 0.01)),
]

recap = pd.DataFrame(controles, columns=["Contrôle", "Attendu", "Obtenu"])
recap["Verdict"] = recap.apply(
    lambda ligne: "OK" if ligne["Obtenu"] == ligne["Attendu"] else "ATTENTION", axis=1
)
recap["Attendu"] = recap["Attendu"].map(joli)
recap["Obtenu"] = recap["Obtenu"].map(joli)
display(recap)

# Le résultat à retenir
resume = pd.DataFrame({
    "Quantité": ["Plancher de référence (graine 42)", "Barre de bruit (écart-type sur 20 découpages)",
                 "Bruit estimé par la formule", f"Seuil de gain ({FACTEUR} x barre)"],
    "Euros": [round(plancher_ref, 2), round(barre_bruit, 2),
              round(erreur_type_formule, 2), round(seuil_gain, 2)],
})
display(resume)

print(f"Règle pour la suite : un essai ne bat le plancher que s'il gagne plus de {seuil_gain:.2f} euros de MAE.")

if (recap["Verdict"] == "OK").all():
    print("Étape 2.2 validée : la barre de bruit est mesurée, les contrôles internes passent.")
else:
    print("ATTENTION : au moins un contrôle interne ne passe pas. Regarder la colonne Verdict.")

# ── Résultat attendu ──
# Un tableau de 7 contrôles, tous en OK, puis le tableau de synthèse avec la barre de bruit en euros
# (à lire sur vos données) et la règle de décision.

,Contrôle,Attendu,Obtenu,Verdict
0,Lignes de train,150654,150654,OK
1,Nombre de découpages,20,20,OK
2,Tailles de validation distinctes,1,1,OK
3,id communs apprentissage / validation (total),0,0,OK
4,Découpages où la médiane bat la moyenne,20,20,OK
5,Barre de bruit strictement positive (1 = oui),1,1,OK
6,MAE moyenne des 20 à moins de 1 % de la MAE su...,1,1,OK


,Quantité,Euros
0,Plancher de référence (graine 42),1810.40
1,Barre de bruit (écart-type sur 20 découpages),6.40
2,Bruit estimé par la formule,13.82
3,Seuil de gain (2 x barre),12.81


Règle pour la suite : un essai ne bat le plancher que s'il gagne plus de 12.81 euros de MAE.
Étape 2.2 validée : la barre de bruit est mesurée, les contrôles internes passent.
